# 03 - Experience B : interference des artefacts

On teste l'hypothese H2 : sur les images generees, la trace de l'insertion est masquee par les artefacts
de generation, ce qui rend le deplacement cover vers stego plus faible que sur le naturel.

La mesure ne demande aucun detecteur entraine. La mesure principale est une distance multivariee entre
les distributions cover et stego, calculee sur les caracteristiques deja en cache. Une mesure illustrative
par moments globaux du residu est aussi fournie, mais elle s'est revelee trop grossiere pour les
algorithmes adaptatifs.

## Configuration

In [ ]:
# ================= CONFIGURATION =================
SOURCES  = ['natural', 'sd', 'sdxl', 'adm']
ALGO     = 'lsb'         # 'lsb', 'uniward' ou 'hill'
PAYLOAD  = 0.4           # charge utile
FEATURE  = 'spam'        # caracteristiques utilisees pour la mesure principale
N        = 300           # images par classe pour la mesure illustrative
SEED     = 42
# ================================================
print('Interference,', ALGO, 'a', PAYLOAD, 'bpp, caracteristiques', FEATURE)

## Environnement

In [ ]:
import os, glob, shutil
import numpy as np
np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/memoire_data'
    ROOT = '/content/corpus'
except Exception:
    DATA_DIR = os.path.abspath('./memoire_data')
    ROOT = os.path.abspath('./corpus')
FEAT_DIR = f'{DATA_DIR}/features_{FEATURE}'
RESULTS = f'{DATA_DIR}/results'
os.makedirs(RESULTS, exist_ok=True)

if not os.path.isdir(f'{ROOT}/natural/cover'):
    zips = sorted(glob.glob(f'{DATA_DIR}/corpus_*.zip'))
    if zips:
        shutil.unpack_archive(zips[-1], ROOT)
print('Corpus :', ROOT, '| caracteristiques :', FEAT_DIR)

In [ ]:
!pip install -q imageio scipy scikit-learn matplotlib
print('Installation terminee.')

## 1. Mesure principale : distance multivariee cover vers stego

On mesure l'ampleur du deplacement entre les distributions cover et stego par la distance de Mahalanobis,
dans l'espace complet des caracteristiques, avec une covariance regularisee. Une distance plus faible sur
une source generee que sur le naturel signale l'interference.

In [ ]:
from sklearn.covariance import LedoitWolf

def distance_interference(src):
    Xc = np.load(f'{FEAT_DIR}/{src}__cover.npy')
    Xs = np.load(f'{FEAT_DIR}/{src}__{ALGO}_p{PAYLOAD}.npy')
    n = min(len(Xc), len(Xs)); Xc, Xs = Xc[:n], Xs[:n]
    both = np.vstack([Xc, Xs])
    mu, sd = both.mean(0), both.std(0) + 1e-9      # meme echelle pour toutes les caracteristiques
    Xc, Xs = (Xc - mu) / sd, (Xs - mu) / sd
    cov = LedoitWolf().fit(np.vstack([Xc, Xs])).covariance_
    diff = Xs.mean(0) - Xc.mean(0)
    return float(np.sqrt(diff @ np.linalg.pinv(cov) @ diff))

distances = {}
for src in SOURCES:
    try:
        distances[src] = distance_interference(src)
        print(f'{src:8s} distance cover vers stego : {distances[src]:.3f}')
    except FileNotFoundError:
        print(f'{src:8s} caracteristiques manquantes, ignore')

if 'natural' in distances:
    print('\nComparaison au naturel :')
    for src in SOURCES:
        if src in distances and src != 'natural':
            r = distances[src] / distances['natural']
            masque = 'masquage' if r < 1 else 'pas de masquage'
            print(f'  {src:8s}: {r:.2f} fois le naturel, {masque}')

## 2. Mesure illustrative : moments globaux du residu

Quatre statistiques globales du residu, comparees entre cover et stego par la taille d'effet de Cohen.
Cette mesure est simple a visualiser, mais trop grossiere pour les algorithmes adaptatifs, dont la trace
ne deplace presque pas ces moments globaux. Elle sert surtout d'illustration.

In [ ]:
import imageio.v2 as imageio
from scipy.signal import convolve2d
from scipy.stats import skew, kurtosis

STATS = ['variance', 'asymetrie', 'kurtosis', 'entropie']

def residu_hf(img):
    k = np.array([[-1, 2, -1], [2, -4, 2], [-1, 2, -1]], float)
    return convolve2d(img, k, mode='same', boundary='symm')

def descripteur(path):
    r = residu_hf(imageio.imread(path).astype(float)).ravel()
    counts, _ = np.histogram(r, bins=64); p = counts / counts.sum(); p = p[p > 0]
    return [float(np.var(r)), float(skew(r)), float(kurtosis(r)), float(-(p * np.log2(p)).sum())]

def descripteurs(dossier, motif, n):
    return np.array([descripteur(f) for f in sorted(glob.glob(f'{dossier}/{motif}'))[:n]])

def cohen_d(a, b):
    s = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return (np.mean(b) - np.mean(a)) / s if s > 0 else 0.0

desc_cover, desc_stego = {}, {}
print(f"{'source':8s} " + ' '.join(f'{n:>10s}' for n in STATS) + '   |d| moyen')
for src in SOURCES:
    c = descripteurs(f'{ROOT}/{src}/cover', '*.pgm', N)
    s = descripteurs(f'{ROOT}/{src}/{ALGO}', f'*_p{PAYLOAD}.pgm', N)
    if len(c) == 0 or len(s) == 0:
        continue
    m = min(len(c), len(s)); desc_cover[src], desc_stego[src] = c[:m], s[:m]
    ds = [abs(cohen_d(c[:m, j], s[:m, j])) for j in range(len(STATS))]
    print(f'{src:8s} ' + ' '.join(f'{d:10.3f}' for d in ds) + f'   {np.mean(ds):.3f}')

## 3. Figure : projection des classes

Projection en deux dimensions des descripteurs de residu, pour voir le chevauchement des classes.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X, couleurs, noms = [], [], []
palette = {'natural': 'tab:blue', 'sd': 'tab:orange', 'sdxl': 'tab:green', 'adm': 'tab:red'}
for src in SOURCES:
    if src not in desc_cover:
        continue
    for classe, D in [('cover', desc_cover[src]), ('stego', desc_stego[src])]:
        X.append(D); couleurs += [palette[src]] * len(D); noms += [f'{src} {classe}'] * len(D)
X = np.vstack(X)
P = PCA(n_components=2, random_state=SEED).fit_transform(StandardScaler().fit_transform(X))

fig, ax = plt.subplots(figsize=(8, 7)); vus = set()
for i in range(len(P)):
    marqueur = 'o' if 'cover' in noms[i] else '^'
    lab = noms[i] if noms[i] not in vus else None; vus.add(noms[i])
    ax.scatter(P[i, 0], P[i, 1], c=couleurs[i], marker=marqueur, s=12, alpha=0.5, label=lab)
ax.set_title('Projection 2D des descripteurs, rond = cover, triangle = stego')
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig(f'{RESULTS}/expB_projection_{ALGO}_p{PAYLOAD}.png', dpi=150); plt.show()

## Interpretation

La mesure principale est la distance multivariee. Une distance plus faible sur une source generee
que sur le naturel signale que l'insertion y est plus masquee, donc de l'interference.

Sur LSB, on observe une distance plus faible sur Stable Diffusion, tandis que SDXL et ADM restent au
niveau du naturel. L'interference se concentre donc sur Stable Diffusion, comme le decalage de domaine.

Limite : les caracteristiques SPAM ne captent pas les algorithmes adaptatifs, donc leur interference
demande SRM a grande echelle. A relancer avec `FEATURE = 'srm'` une fois ces caracteristiques extraites.